# OMERO annotation workflow for screens, plates and wells

This workflow provides guidance through the process of metadata collection during an bioimaging experiment via Adamant, the annotation of data in OMERO and the analysis of images in the end.\
<span style="color:red">
Change the links to Adamant, eLabFTW and OMERO to your specific instances
</span>
1. Do your experiment
2. Fill the biological metadata into the REMBI compatible [Excel template]()
3. Enter experiment metadata in [Adamant](https://plasma-mds.github.io/adamant/)\
    2.1 Click on "BROWSE SCHEMA"\
    2.2 Load the [Screen_schema.json](https://github.com/INP-SDT/Jupyter4OMERO/blob/main/metadataSchemas/screenSchemas/Screen_schema.json) into Adamant\
    2.3 Click on "RENDER"\
    2.4 Fill out all information from your experiment\
    2.5 Click on "PROCEDE" (in the bottom right corner of the HTML form) to send the metadata to the [eLabFTW]()  \
    <span style="color:red"> An eLabFTW API key is necessary therefore a previos login into the ELN is necessary </span>
4. Go to [OMERO](http://pm-omero.intranet.inp-greifswald.de:4080/webclient/login/?url=%2Fwebclient%2F)
    3.1 Enter your credentials to log in  

### Annotation of an OMERO Screen
4. Create a new "Screen"
5. Start the software OMERO.insight ([download available here](https://www.openmicroscopy.org/omero/downloads/))
6. Upload the taken images (from your experiment) into the in step 4. created Screen via OMERO.insight
7. Run the following code to anotate your images with the previously (step 2) collected metadata

In [1]:
# import all important libraries containing the needed functionalities for the workflow
import elabFTW_Api_handler as ELN
import OMERO_handler as OME

import ipywidgets as widgets
from IPython.display import display, clear_output

### eLabFTW log-in
Direct input of the api key.\
<span style="color:red">
Run the following line of code only once to generate the entry field. Afterwards you only have to fill this field and run the later code to read the input properly.
</span>

In [2]:
# ask for api key input
api = widgets.Password(description="eLab API key:", layout=widgets.Layout(width='300px'), style={'description_width': 'initial'})
display(api)

Password(description='eLab API key:', layout=Layout(width='300px'), style=TextStyle(description_width='initial…

In [4]:
# check validity of the API key and setup all the API instances
spinner = widgets.HTML('<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')
display(spinner)

ELN.API_configurator_direct_input(api.value)

spinner.value = '<span style="color:green; font-weight:bold;">✅ Done!</span>'

HTML(value='<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')

API key is valid.
Access to eLabFTW granted


### OMERO log-in credentials
Initalize the input fields for your OMERO login credentials.\
<span style="color:red">
Run the following line of code only once to generate the entry field. Afterwards you only have to fill this field and run the later code to read the input properly.
</span>

In [3]:
# display fields to enter OMERO credentials
usrname = widgets.Text(description="OMERO username:", layout=widgets.Layout(width='300px'), style={'description_width': 'initial'})
passwrd = widgets.Password(description="OMERO password:", layout=widgets.Layout(width='300px'), style={'description_width': 'initial'})
display(usrname)
display(passwrd)

Text(value='', description='OMERO username:', layout=Layout(width='300px'), style=TextStyle(description_width=…

Password(description='OMERO password:', layout=Layout(width='300px'), style=TextStyle(description_width='initi…

In [4]:
# log into OMERO with given credentials
spinner = widgets.HTML('<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')
display(spinner)

if usrname.value and passwrd.value:
    connection = OME.Omero_login_Jupyter(usrname.value, passwrd.value)
else:
    print("Enter credentials and rerun this line to log into OMERO")

spinner.value = '<span style="color:green; font-weight:bold;">✅ Done!</span>'

HTML(value='<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')

Enter credentials and rerun this line to log into OMERO


### Annotation of screens, paltes and wells
8. Provide information needed for the annotation of the screen and plate and wells (if wanted).
   
<span style="color:red">
Run the following line of code only once to generate the entry field. Afterwards you only have to fill this field and run the later code to read the input properly.
</span>

In [5]:
# display fields for annotation information
screen = widgets.Text(description="Name of the screen that shall be annotated:", layout=widgets.Layout(width='600px'), style={'description_width': 'initial'})
plate = widgets.Text(description="Name of the plate that shall be annotated:", layout=widgets.Layout(width='600px'), style={'description_width': 'initial'})
experimentMetadata = widgets.Text(description="ID of the eLab experiment containing the metadata:", layout=widgets.Layout(width='600px'), style={'description_width': 'initial'})
checkbox = widgets.Checkbox(
    value=False,
    description='Annotate wells?'
)
output = widgets.Output()
# define output on checkbox click
def on_checkbox_change(change):
    with output:
        clear_output()  # vorherige Ausgabe löschen
        if change['new']:
            print("Wells will be annotated")
        else:
            print("Wells will not be annotated")

checkbox.observe(on_checkbox_change, names='value')
# display all widgets
display(screen)
display(plate)
display(experimentMetadata)
display(checkbox, output)

Text(value='', description='Name of the screen that shall be annotated:', layout=Layout(width='600px'), style=…

Text(value='', description='Name of the plate that shall be annotated:', layout=Layout(width='600px'), style=T…

Text(value='', description='ID of the eLab experiment containing the metadata:', layout=Layout(width='600px'),…

Checkbox(value=False, description='Annotate wells?')

Output()

In [6]:
# annotate metadata from eLab to Omero plate and screen
spinner = widgets.HTML('<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')
display(spinner)
# set data access to group "users" 
connection.setGroupForSession(3)
metadata = ELN.extract_experiment_attachments(expID = int(experimentMetadata.value), filetype = "JSON metadata")
bioData = ELN.extract_experiment_attachments(expID = int(experimentMetadata.value), filetype = "Excel file")

OME.annotate_screen(conn = connection, screenName = screen.value, jsonData = metadata)
OME.annotate_plate(conn = connection, plateName = plate.value, jsonData = metadata, excelData = bioData, wellBool = checkbox.value)

spinner.value = '<span style="color:green; font-weight:bold;">✅ Done!</span>'

HTML(value='<i class="fa fa-spinner fa-spin fa-2x fa-fw"></i> Code processing...')

NameError: name 'connection' is not defined

Well done, your annotation was successful, if you want to annotate another screen or plate just change the fields at the beginning of this section and rerun the last code line!